# TP Sismique - Viking Graben (Version simplifiée qui marche)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import convolve
%matplotlib inline

In [ ]:
# Création des données synthétiques réalistes
dt = 0.004
time = np.arange(0, 2.0, dt)
ntraces = 100
offsets = np.arange(ntraces) * 25

# Ondelette
t_w = np.arange(-0.1, 0.1, dt)
wavelet = (1 - 2*(np.pi*25*t_w)**2) * np.exp(-(np.pi*25*t_w)**2)

# Réflecteurs typiques d'un graben
reflectors = [
    (0.20, 0.3),   # temps, amplitude
    (0.45, -0.2),
    (0.70, 0.4),
    (0.95, -0.3),
    (1.20, 0.25),
    (1.50, -0.2),
    (1.75, 0.15)
]

# Construction du gather
gather = []
v_rms = 1800

for offset in offsets:
    trace = np.zeros_like(time)
    for t0, amp in reflectors:
        t = np.sqrt(t0**2 + (offset / v_rms)**2)
        idx = int(t / dt)
        if idx < len(trace):
            trace[idx] = amp
    trace = convolve(trace, wavelet, mode='same')
    trace += np.random.randn(len(time)) * 0.05
    gather.append(trace)

gather = np.array(gather).T

print(f"Dimensions: {gather.shape}")
plt.figure(figsize=(12,8))
plt.imshow(gather, aspect='auto', cmap='seismic', extent=[0, ntraces, time[-1], 0])
plt.colorbar()
plt.title('Données synthétiques - Style Viking Graben')
plt.xlabel('Trace / CDP')
plt.ylabel('Temps (s)')
plt.show()

In [ ]:
# Filtrage
from scipy.signal import butter, filtfilt

def bandpass(data, dt, f_low, f_high):
    nyquist = 0.5 / dt
    b, a = butter(4, [f_low/nyquist, f_high/nyquist], btype='band')
    return filtfilt(b, a, data, axis=0)

data_filt = bandpass(gather, dt, 8, 60)

plt.figure(figsize=(12,8))
plt.imshow(data_filt, aspect='auto', cmap='seismic', extent=[0, ntraces, time[-1], 0])
plt.title('Après filtrage 8-60 Hz')
plt.xlabel('Trace')
plt.ylabel('Temps (s)')
plt.show()

In [ ]:
# Correction NMO
def apply_nmo(data, time, offsets, v_rms):
    data_nmo = np.zeros_like(data)
    dt = time[1] - time[0]
    for itrace, offset in enumerate(offsets):
        for it, t0 in enumerate(time):
            t = np.sqrt(t0**2 + (offset / v_rms)**2)
            idx = int(t / dt)
            if idx < len(time):
                data_nmo[it, itrace] = data[idx, itrace]
    return data_nmo

data_nmo = apply_nmo(data_filt, time, offsets, v_rms)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
axes[0].imshow(data_filt, aspect='auto', cmap='seismic', extent=[0, ntraces, time[-1], 0])
axes[0].set_title('Avant NMO')
axes[1].imshow(data_nmo, aspect='auto', cmap='seismic', extent=[0, ntraces, time[-1], 0])
axes[1].set_title('Après NMO')
plt.show()

In [ ]:
# Empilement
stack = np.mean(data_nmo, axis=1)

plt.figure(figsize=(12, 8))
plt.plot(stack, time, 'k-', linewidth=1.5)
plt.ylim(time[-1], 0)
plt.xlabel('Amplitude')
plt.ylabel('Temps (s)')
plt.title('Trace empilée - Style Viking Graben')
plt.grid(True)
plt.show()

In [ ]:
# Interprétation
plt.figure(figsize=(14, 10))
plt.imshow(data_nmo, aspect='auto', cmap='seismic', extent=[0, ntraces, time[-1], 0], vmin=-0.3, vmax=0.3)

# Annotation des failles
plt.axvline(x=30, color='red', linestyle='--', linewidth=2, label='Faille Ouest')
plt.axvline(x=70, color='red', linestyle='--', linewidth=2, label='Faille Est')

# Horizons
for t in [0.25, 0.5, 0.75, 1.0, 1.3, 1.6]:
    plt.axhline(y=t, color='cyan', linestyle='-', linewidth=1, alpha=0.5)

plt.fill_betweenx([0, time[-1]], 30, 70, alpha=0.2, color='blue', label='Graben')
plt.legend()
plt.title('Interprétation - Graben avec failles listriques')
plt.xlabel('CDP')
plt.ylabel('Temps (s)')
plt.show()

print("""
✅ TP terminé !
Ce que vous avez fait :
- Génération de données synthétiques réalistes
- Filtrage passe-bande
- Correction NMO
- Empilement
- Interprétation géologique (failles, graben)
""")